# Gold Layer (Business Aggregations)
In this notebook, we will read the clean data from the `silver` layer, perform business aggregations, and write the final results to the `gold` layer.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, avg

print("Starting Spark Session...")
spark = SparkSession.builder \
    .appName("DataCore_Gold") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,io.unitycatalog:unitycatalog-spark_2.12:0.2.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "io.unitycatalog.spark.UCSingleCatalog") \
    .config("spark.sql.catalog.unity", "io.unitycatalog.spark.UCSingleCatalog") \
    .config("spark.sql.catalog.unity.uri", "http://unitycatalog:8080") \
    .config("spark.sql.defaultCatalog", "unity") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark Session Ready!")

Starting Spark Session...
Spark Session Ready!


In [2]:
# 1. Read the clean data from Silver 
print("Reading from Silver...")
s3_silver_path = "s3://silver/users_clean"
df_silver = spark.read.format("delta").load(s3_silver_path)
df_silver.show()

# 2. Perform Business Aggregation (The Gold Layer)
# Example: Count users and find average age
df_gold = df_silver.groupBy().agg(
    count("Name").alias("Total_Users"),
    avg("Age").alias("Average_Age")
)

print("Aggregated Gold Data:")
df_gold.show()

Reading from Silver...
+-----+---+--------------------+
| Name|Age|      ingestion_time|
+-----+---+--------------------+
|Ahmed| 25|2026-09-29 12:24:...|
| Omar| 30|2026-09-29 12:24:...|
+-----+---+--------------------+

Aggregated Gold Data:
+-----------+-----------+
|Total_Users|Average_Age|
+-----------+-----------+
|          2|       27.5|
+-----------+-----------+



In [3]:
# 3. Write to MinIO directly 
print("Writing to Gold Bucket...")
s3_gold_path = "s3://gold/user_stats"

df_gold.write.format("delta") \
    .mode("overwrite") \
    .save(s3_gold_path)

print("Success! Gold Table is now ready for BI Dashboards.")

Writing to Gold Bucket...
Success! Gold Table is now ready for BI Dashboards.
